# Stage 7C: public package OOF audit

This standalone Colab notebook inventories the fleongg and pilkwang public model packages. It checks for true OOF assets, row IDs, fold metadata, and manifests before any blend is attempted. It does not train or submit.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
python = repo_dir / '.venv/bin/python'
subprocess.run(['uv', 'pip', 'install', '--python', str(python), 'kagglehub'], check=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)

In [ ]:
def download_dataset(handle):
    code = f"import kagglehub; print('DATASET_PATH=' + kagglehub.dataset_download({handle!r}))"
    result = subprocess.run([str(python), '-c', code], check=True, capture_output=True, text=True)
    print(result.stdout)
    line = [value for value in result.stdout.splitlines() if value.startswith('DATASET_PATH=')][-1]
    return Path(line.split('=', 1)[1])

fleongg_dir = download_dataset('fleongg/rogii-claude-models-pub')
pilkwang_dir = download_dataset('pilkwang/rogii-model-package')
print('fleongg:', fleongg_dir)
print('pilkwang:', pilkwang_dir)

In [ ]:
BASE_RUN_ID = 'stage7_public_residual_gate_full_v001'
base_run = artifact_dir / BASE_RUN_ID
assert (base_run / 'base_oof.parquet').is_file(), f'Missing: {base_run / "base_oof.parquet"}'
report_path = artifact_dir / 'stage7c_public_package_audit_v001.json'
subprocess.run([
    'uv', 'run', 'rogii-public-package-audit',
    '--base-run', str(base_run),
    '--package', f'fleongg={fleongg_dir}',
    '--package', f'pilkwang={pilkwang_dir}',
    '--output', str(report_path),
], cwd=repo_dir, check=True)

In [ ]:
report = json.loads(report_path.read_text())
summary = {}
for package in report['packages']:
    summary[package['label']] = {
        'file_count': package['file_count'],
        'total_gib': package['total_bytes'] / 1024**3,
        'interesting_paths': [row['path'] for row in package['candidate_files']],
        'candidate_details': package['candidate_files'],
        'json_files': list(package['json_documents']),
        'json_documents': package['json_documents'],
        'errors': package['errors'],
    }
summary

Send the final `summary` dictionary back. The next blend notebook will be generated only after the OOF file, prediction column, ID ordering, and fold provenance are identified. Files with matching row counts but no ID or manifest proof are not accepted automatically.